# NTX Q1 Stage 19 — Direct Baselines and Component Ablations

Runs the comparison that most directly supports the paper's novelty: single-best interpretation versus set-valued authorization, plus policy-only, Anchor-only, clarification-only, and full NiyamTrace variants on the same frozen case records.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

In [ ]:
import pandas as pd, numpy as np
STAGE="19_baselines"
OUT=ensure_dir(RESULTS_ROOT/STAGE)
INP=ensure_dir(ROOT/"inputs"/STAGE)

# Preferred input: a per-case frozen log with expected_verdict and the replayed stage verdicts.
candidates=[
    Path(os.environ.get("NTX_FROZEN_CASES","")),
    ROOT/"repo/experiments/results/frozen_cases.csv",
]
path=next((p for p in candidates if str(p) and p.exists()),None)
if path is None:
    # Write exact schema needed. No synthetic substitute is used for publication metrics.
    pd.DataFrame(columns=[
        "case_id","variant_group_id","language","relation","expected_verdict",
        "raw_policy_verdict","single_best_verdict","policy_only_verdict",
        "anchor_only_verdict","clarification_only_verdict","set_valued_verdict","full_verdict"
    ]).to_csv(OUT/"frozen_cases_REQUIRED_SCHEMA.csv",index=False)
    record_status(OUT,STAGE,"WAITING_FOR_FROZEN_CASE_LOG")
    print("Provide NTX_FROZEN_CASES pointing to a case-level CSV.")
else:
    df=pd.read_csv(path)
    print("Loaded",len(df),"rows from",path)

In [ ]:
if path is not None:
    variants={
        "raw":"raw_policy_verdict",
        "single_best":"single_best_verdict",
        "policy_only":"policy_only_verdict",
        "anchor_only":"anchor_only_verdict",
        "clarification_only":"clarification_only_verdict",
        "set_valued":"set_valued_verdict",
        "full":"full_verdict",
    }
    rows=[]
    for name,col in variants.items():
        if col not in df.columns: 
            rows.append({"variant":name,"status":"MISSING_COLUMN","column":col})
            continue
        tmp=df[["expected_verdict",col]].rename(columns={col:"actual_verdict"}).dropna()
        m=classification_metrics(tmp)
        rows.append({"variant":name,"status":"OK",**m})
    metrics=pd.DataFrame(rows)
    metrics.to_csv(OUT/"baseline_metrics.csv",index=False)
    display(metrics)

In [ ]:
# Paired bootstrap differences full vs single-best and full vs policy-only.
if path is not None:
    comparisons=[]
    rng=np.random.default_rng(SEED)
    for base_name,base_col in [("single_best","single_best_verdict"),("policy_only","policy_only_verdict"),("raw","raw_policy_verdict")]:
        if base_col not in df.columns or "full_verdict" not in df.columns: continue
        x=df[["expected_verdict",base_col,"full_verdict"]].dropna().copy()
        x["base_correct"]=x[base_col].map(normalize_verdict)==x.expected_verdict.map(normalize_verdict)
        x["full_correct"]=x.full_verdict.map(normalize_verdict)==x.expected_verdict.map(normalize_verdict)
        x["delta"]=x.full_correct.astype(int)-x.base_correct.astype(int)
        mean,lo,hi=bootstrap_ci(x.delta,B=10000,seed=SEED)
        e=x.expected_verdict.map(normalize_verdict)
        unsafe_base=((x[base_col].map(normalize_verdict)=="ALLOW")&(e!="ALLOW")).sum()
        unsafe_full=((x.full_verdict.map(normalize_verdict)=="ALLOW")&(e!="ALLOW")).sum()
        comparisons.append({
            "comparison":f"full-minus-{base_name}",
            "accuracy_delta":mean,"ci95_low":lo,"ci95_high":hi,
            "unsafe_allow_base_n":int(unsafe_base),"unsafe_allow_full_n":int(unsafe_full),
            "n":len(x)
        })
    comp=pd.DataFrame(comparisons)
    comp.to_csv(OUT/"paired_comparisons.csv",index=False)
    display(comp)
    paper_ready=any(comp.comparison.eq("full-minus-single_best")) and len(df)>=1000
    record_status(OUT,STAGE,"PASS" if paper_ready else "PARTIAL",paper_ready=bool(paper_ready))

In [ ]:
z=package_stage(OUT,"NTX_Q1_19_BASELINES_ABLATIONS_RESULTS.zip")
print("Created:",z)
maybe_colab_download(z)